In [ ]:
# STEP 1 - Install packages
!pip install mediapipe opencv-python numpy

In [ ]:
# STEP 2 - Imports
import cv2
import mediapipe as mp
import numpy as np
import math
from enum import Enum
import time
import tkinter as tk
from tkinter import simpledialog

In [ ]:
# STEP 3 - Initialize MediaPipe Pose
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
pose = mp_pose.Pose(min_detection_confidence=0.6, min_tracking_confidence=0.5)

In [ ]:
# STEP 4 -Import our TTS system
import sys
import os
sys.path.append('.')  # Add current directory to path
from tts_final import FullMP3TTS

# Initialize TTS system
tts_system = FullMP3TTS()

def speak(text):
    """Main speak function - now uses our MP3 TTS system"""
    print(f"Speaking: {text}")

In [ ]:
# TEST CELL
import sys
sys.path.append('.')
from tts_final import FullMP3TTS

tts_system = FullMP3TTS()

def speak(text):
    print(f"Speaking: {text}")

print("✅ TTS system ready!")

In [ ]:
# Test our TTS system
print("Testing TTS integration...")
speak("Hello from the integrated TTS system")
tts_system.play_countdown()
print("TTS test complete!")

In [ ]:
# TEST CALL
def start_test_countdown():
    """Display countdown before starting the test"""
    import time
    
    countdown_messages = [
        ("Get Ready: 3", 'countdown_3'),
        ("Get Ready: 2", 'countdown_2'), 
        ("Get Ready: 1", 'countdown_1'),
        ("GO!", 'countdown_go')
    ]
    
    for message, audio_key in countdown_messages:
        print(f"\n{message}")
        tts_system.play_audio(audio_key)
        time.sleep(1)
    
    print("Test starting...")

In [ ]:
start_test_countdown()

In [ ]:
# STEP 5 - Debug function landmark
def debug_print_landmarks(landmarks):
    """
    Print all relevant landmark positions for debugging
    """
    print("\n--- DEBUG: Landmark Positions ---")

    # Define key landmarks for this exercise
    key_points = {
        'LEFT_HIP': mp_pose.PoseLandmark.LEFT_HIP.value,
        'RIGHT_HIP': mp_pose.PoseLandmark.RIGHT_HIP.value,
        'LEFT_KNEE': mp_pose.PoseLandmark.LEFT_KNEE.value,
        'RIGHT_KNEE': mp_pose.PoseLandmark.RIGHT_KNEE.value,
        'LEFT_ANKLE': mp_pose.PoseLandmark.LEFT_ANKLE.value,
        'RIGHT_ANKLE': mp_pose.PoseLandmark.RIGHT_ANKLE.value,
        'LEFT_HEEL': mp_pose.PoseLandmark.LEFT_HEEL.value,
        'RIGHT_HEEL': mp_pose.PoseLandmark.RIGHT_HEEL.value,
        'LEFT_FOOT_INDEX': mp_pose.PoseLandmark.LEFT_FOOT_INDEX.value,
        'RIGHT_FOOT_INDEX': mp_pose.PoseLandmark.RIGHT_FOOT_INDEX.value,
        'LEFT_SHOULDER': mp_pose.PoseLandmark.LEFT_SHOULDER.value,
        'RIGHT_SHOULDER': mp_pose.PoseLandmark.RIGHT_SHOULDER.value,
        'LEFT_WRIST': mp_pose.PoseLandmark.LEFT_WRIST.value,
        'RIGHT_WRIST': mp_pose.PoseLandmark.RIGHT_WRIST.value,
        'LEFT_INDEX': mp_pose.PoseLandmark.LEFT_INDEX.value,
        'RIGHT_INDEX': mp_pose.PoseLandmark.RIGHT_INDEX.value,
    }

    for name, idx in key_points.items():
        landmark = landmarks[idx]
        print(f"{name}: x={landmark.x:.3f}, y={landmark.y:.3f}, z={landmark.z:.3f}, visibility={landmark.visibility:.3f}")

    # Calculate and print some useful measurements
    print("\n--- Calculated Values ---")

    # Hip height average
    left_hip_y = landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y
    right_hip_y = landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y
    avg_hip_y = (left_hip_y + right_hip_y) / 2
    print(f"Average Hip Height (Y): {avg_hip_y:.3f}")

    # Knee height average
    left_knee_y = landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y
    right_knee_y = landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y
    avg_knee_y = (left_knee_y + right_knee_y) / 2
    print(f"Average Knee Height (Y): {avg_knee_y:.3f}")

    # Ankle positions
    left_ankle_y = landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y
    right_ankle_y = landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y
    print(f"Left Ankle Y: {left_ankle_y:.3f}, Right Ankle Y: {right_ankle_y:.3f}")

    # Hip to knee distance (normalized)
    hip_knee_diff = avg_hip_y - avg_knee_y
    print(f"Hip-Knee Height Difference: {hip_knee_diff:.3f}")

    print("-" * 40)

In [ ]:
# STEP 6 - Test States & Flexibility Class (Define states for the flexibility test)
class TestState(Enum):
    FULL_BODY_DETECTION = 0
    SITTING_CHECK = 1
    FOOT_FLAT_CHECK = 2
    LEG_EXTENDED_CHECK = 3
    HANDS_POSITION_CHECK = 4
    INHALE_PREPARATION = 5
    FORWARD_REACH = 6
    HOLD_POSITION = 7
    MEASUREMENT = 8
    COMPLETE = 9

class FlexibilityTest:
    def __init__(self):
        self.current_state = TestState.FULL_BODY_DETECTION
        self.state_passed = False
        self.hold_timer = 0
        self.hold_start_time = None
        self.measurement_result = None
        self.debug_enabled = False
        self.advance = False

        # NEW: phase control
        self.phase = "preview"  # "instruction" or "validation"
        self.phase_start_time = None

    def start_phase(self, phase):
        self.phase = phase
        self.phase_start_time = time.time()

    def get_preview_message(self):
        """Message shown before each step — tells user what’s coming next."""
        messages = {
            TestState.FULL_BODY_DETECTION: "Be ready to sit or stand",
            TestState.SITTING_CHECK: "Be ready to sit correctly",
            TestState.FOOT_FLAT_CHECK: "Be ready to keep one foot flat",
            TestState.LEG_EXTENDED_CHECK: "Be ready to straighten your leg",
            TestState.HANDS_POSITION_CHECK: "Be ready to place hands on each other",
            TestState.FORWARD_REACH: "Inhale and be ready to exhale and reach forward",
            TestState.MEASUREMENT: "Hold and be ready for flexibility measurement",
            TestState.COMPLETE: "Test complete"
        }
        return messages.get(self.current_state, "")

    def get_state_message(self):
        """Get instruction message for current state"""
        messages = {
            TestState.FULL_BODY_DETECTION: "Please stand or sit fully in camera frame",
            TestState.SITTING_CHECK: "Please sit on the edge of a chair",
            TestState.FOOT_FLAT_CHECK: "Place one foot flat on the floor",
            TestState.LEG_EXTENDED_CHECK: "Please extend other leg forward",
            TestState.HANDS_POSITION_CHECK: "Please place one hand on top of the other",
            TestState.FORWARD_REACH: "Exhale and reach forward toward your toes",
            TestState.MEASUREMENT: "Measuring flexibility...",
            TestState.COMPLETE: f"Test complete! Result: {self.measurement_result}"
        }
        return messages.get(self.current_state, "")

    def advance_state(self):
        """Move to the next state"""
        if self.current_state == TestState.FULL_BODY_DETECTION:
            self.current_state = TestState.SITTING_CHECK
        elif self.current_state == TestState.SITTING_CHECK:
            self.current_state = TestState.FOOT_FLAT_CHECK
        elif self.current_state == TestState.FOOT_FLAT_CHECK:
            self.current_state = TestState.LEG_EXTENDED_CHECK
        elif self.current_state == TestState.LEG_EXTENDED_CHECK:
            self.current_state = TestState.HANDS_POSITION_CHECK
        elif self.current_state == TestState.HANDS_POSITION_CHECK:
            self.current_state = TestState.FORWARD_REACH
        elif self.current_state == TestState.FORWARD_REACH:
            self.current_state = TestState.MEASUREMENT
        elif self.current_state == TestState.MEASUREMENT:
            self.current_state = TestState.COMPLETE

    def reset_test(self):
        """Reset to beginning"""
        self.current_state = TestState.FULL_BODY_DETECTION
        self.state_passed = False
        self.hold_timer = 0
        self.hold_start_time = None
        self.measurement_result = None
        self.phase = "preview"  # "instruction" or "validation"
        self.phase_start_time = None

    def measure_test(self):
        self.current_state = TestState.MEASUREMENT
        self.state_passed = False
        self.hold_timer = 0
        self.hold_start_time = None
        self.measurement_result = None
        self.phase = "preview"  # "instruction" or "validation"
        self.phase_start_time = None

In [ ]:
# STEP 7 - check_full_body 
def check_full_body(landmarks):
    """
    Ensure the full person is detected before starting test.
    Requires key joints (shoulders, hips, knees, ankles, feet) with good visibility.
    """
    key_points = [
        mp_pose.PoseLandmark.LEFT_SHOULDER,
        mp_pose.PoseLandmark.RIGHT_SHOULDER,
        mp_pose.PoseLandmark.LEFT_HIP,
        mp_pose.PoseLandmark.RIGHT_HIP,
        mp_pose.PoseLandmark.LEFT_KNEE,
        mp_pose.PoseLandmark.RIGHT_KNEE,
        mp_pose.PoseLandmark.LEFT_ANKLE,
        mp_pose.PoseLandmark.RIGHT_ANKLE,
        mp_pose.PoseLandmark.LEFT_FOOT_INDEX,
        mp_pose.PoseLandmark.RIGHT_FOOT_INDEX,
    ]

    VISIBLE_THRESHOLD = 0.6
    visible_count = 0

    for kp in key_points:
        if landmarks[kp.value].visibility > VISIBLE_THRESHOLD:
            visible_count += 1

    if visible_count == len(key_points):
        return True, "Full body detected"
    else:
        return False, "Position yourself fully in camera view"


In [ ]:
# STEP 8 - check_sitting_position
def check_sitting_position(landmarks):
    """Check if person is sitting based on hip and knee angles"""
    left_hip = landmarks[mp_pose.PoseLandmark.LEFT_HIP.value]
    right_hip = landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value]
    left_knee = landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value]
    right_knee = landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value]

    avg_hip_y = (left_hip.y + right_hip.y) / 2
    avg_knee_y = (left_knee.y + right_knee.y) / 2
    hip_knee_distance = abs(avg_hip_y - avg_knee_y)

    if hip_knee_distance < 0.09:
        return True, "Sitting position detected"
    else:
        return False, "Not sitting - please sit on edge of chair"

In [ ]:
# STEP 9 - check_foot_flat
def check_foot_flat_on_floor(landmarks):
    """
    Checks if at least one foot is flat by comparing heel and foot index Y positions.
    """
    Y_TOL = 0.06  # tolerance for vertical alignment

    def foot_flat(heel, toe):
        return abs(heel.y - toe.y) < Y_TOL

    # Left foot
    left_flat = foot_flat(
        landmarks[mp_pose.PoseLandmark.LEFT_HEEL.value],
        landmarks[mp_pose.PoseLandmark.LEFT_FOOT_INDEX.value]
    )

    # Right foot
    right_flat = foot_flat(
        landmarks[mp_pose.PoseLandmark.RIGHT_HEEL.value],
        landmarks[mp_pose.PoseLandmark.RIGHT_FOOT_INDEX.value]
    )

    if left_flat or right_flat:
        return True, "Foot is flat on the floor"
    else:
        return False, "Please place one foot flat on the floor"

In [ ]:
# STEP 10 - calculate_angle & check_leg_extended
def calculate_angle_3d(a, b, c):
    """
    Calculates the angle at point b in 3D given points a, b, c.
    Each point is an object with .x, .y, .z attributes.
    """
    # Vectors BA and BC
    ba = (a.x - b.x, a.y - b.y, a.z - b.z)
    bc = (c.x - b.x, c.y - b.y, c.z - b.z)

    # Dot product and magnitudes
    dot_product = ba[0]*bc[0] + ba[1]*bc[1] + ba[2]*bc[2]
    mag_ba = math.sqrt(ba[0]**2 + ba[1]**2 + ba[2]**2)
    mag_bc = math.sqrt(bc[0]**2 + bc[1]**2 + bc[2]**2)

    if mag_ba == 0 or mag_bc == 0:
        return 0

    angle_rad = math.acos(max(min(dot_product / (mag_ba * mag_bc), 1.0), -1.0))
    return math.degrees(angle_rad)

def check_leg_extended(landmarks):
    """
    Checks if either leg is straight using 3D hip-knee-ankle angle.
    """
    lh, lk, la = landmarks[mp_pose.PoseLandmark.LEFT_HIP.value], landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value], landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value]
    rh, rk, ra = landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value], landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value], landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value]

    left_knee_angle = calculate_angle_3d(lh, lk, la)
    right_knee_angle = calculate_angle_3d(rh, rk, ra)

    ANGLE_THRESHOLD = 155  # degrees - Updated from 80° to ensure proper leg extension per Rikli & Jones protocol

    if left_knee_angle > ANGLE_THRESHOLD:
        return True, f"leg is straight"
    elif right_knee_angle > ANGLE_THRESHOLD:
        return True, f"leg is straight"
    else:
        return False, f"Please extend one leg straight"

In [ ]:
# STEP 11 - check_hands_position
def check_hands_position(landmarks):
    """
    Rikli & Jones protocol: palms down, middle fingers even.
    Checks:
    1. Both wrists at same Y level (middle fingers even)
    2. Both wrists close horizontally (hands together, not spread apart)
    3. Wrists below shoulders (hands in lap/chest area)
    """
    left_wrist     = landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value]
    right_wrist    = landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value]
    left_shoulder  = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value]
    right_shoulder = landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value]

    # 1. Wrists at same height (Y axis) — middle fingers even
    wrist_y_diff = abs(left_wrist.y - right_wrist.y)
    fingers_even = wrist_y_diff < 0.08

    # 2. Wrists close horizontally (X axis) — hands together
    wrist_x_diff = abs(left_wrist.x - right_wrist.x)
    hands_together = wrist_x_diff < 0.25

    # 3. Wrists below shoulders — hands in lap/chest, not raised
    avg_shoulder_y = (left_shoulder.y + right_shoulder.y) / 2
    avg_wrist_y    = (left_wrist.y + right_wrist.y) / 2
    hands_below_shoulders = avg_wrist_y > avg_shoulder_y

    if fingers_even and hands_together and hands_below_shoulders:
        return True, "Hands positioned correctly"
    elif not fingers_even:
        return False, "Align your middle fingers at the same level"
    elif not hands_together:
        return False, "Bring your hands closer together"
    else:
        return False, "Lower your hands to lap level"

In [ ]:
# STEP 12 - check_inhale
def check_inhale(landmarks):
    """
    Instructs the subject to inhale in preparation for forward reach.
    Always passes after showing the instruction.
    """
    return True, "Inhale deeply... next, as you exhale, you'll reach forward."

In [ ]:
# STEP 13 - check_forward_reach
def check_forward_reach(landmarks):
    """
    Check if the person is reaching forward toward the extended foot.
    Uses only horizontal alignment with ankle — camera angle makes
    vertical (Y) check unreliable from side view.
    """
    lw = landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value]
    rw = landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value]
    hand_x = (lw.x + rw.x) / 2

    lk = landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value]
    rk = landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value]
    la = landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value]
    ra = landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value]

    left_angle  = calculate_angle_3d(landmarks[mp_pose.PoseLandmark.LEFT_HIP.value],  lk, la)
    right_angle = calculate_angle_3d(landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value], rk, ra)

    extended_ankle = la if left_angle > right_angle else ra

    x_diff = abs(hand_x - extended_ankle.x)
    hands_near_foot = x_diff < 0.2

    if hands_near_foot:
        return True, "Reaching forward toward extended leg"
    else:
        return False, "Reach forward toward your extended foot"

In [ ]:
def measure_flexibility_old(landmarks):
    """
    Simple flexibility measure:
    Distance between fingertips and extended foot toes.
    Positive = past toes, Negative = short of toes.
    """
    # Fingertips (average of index fingers)
    left_index = landmarks[mp_pose.PoseLandmark.LEFT_INDEX.value]
    right_index = landmarks[mp_pose.PoseLandmark.RIGHT_INDEX.value]
    fingertip_y = (left_index.y + right_index.y) / 2

    # Detect extended leg (straighter knee)
    lh, lk, la = (landmarks[mp_pose.PoseLandmark.LEFT_HIP.value],
                  landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value],
                  landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value])
    rh, rk, ra = (landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value],
                  landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value],
                  landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value])

    left_knee_angle = calculate_angle_3d(lh, lk, la)
    right_knee_angle = calculate_angle_3d(rh, rk, ra)

    if left_knee_angle > right_knee_angle:
        toe = landmarks[mp_pose.PoseLandmark.LEFT_FOOT_INDEX.value]
        print('left')
    else:
        toe = landmarks[mp_pose.PoseLandmark.RIGHT_FOOT_INDEX.value]
        print('right')

    # Difference in normalized coords
    diff_norm = toe.y - fingertip_y

    # Assume normalized 0.1 ≈ 10 cm (simple scale, adjustable)
    diff_cm = diff_norm * 100  

    if diff_cm >= 0:
        return f"{diff_cm:.1f} cm past toes"
    else:
        return f"{abs(diff_cm):.1f} cm short of toes"


In [7]:
# STEP 14 - measure_flexibility
def measure_flexibility(landmarks):
    # Fingertips (average of index fingers)
    left_index = landmarks[mp_pose.PoseLandmark.LEFT_INDEX.value]
    right_index = landmarks[mp_pose.PoseLandmark.RIGHT_INDEX.value]
    fingertip = np.array([
        (left_index.x + right_index.x)/2,
        (left_index.y + right_index.y)/2,
        (left_index.z + right_index.z)/2
    ])

    # Detect extended leg
    lh, lk, la = (landmarks[mp_pose.PoseLandmark.LEFT_HIP.value],
                  landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value],
                  landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value])
    rh, rk, ra = (landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value],
                  landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value],
                  landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value])

    left_knee_angle = calculate_angle_3d(lh, lk, la)
    right_knee_angle = calculate_angle_3d(rh, rk, ra)

    if left_knee_angle > right_knee_angle:
        toe_landmark = landmarks[mp_pose.PoseLandmark.LEFT_FOOT_INDEX.value]
        leg_side = "Left"
    else:
        toe_landmark = landmarks[mp_pose.PoseLandmark.RIGHT_FOOT_INDEX.value]
        leg_side = "Right"

    toe = np.array([toe_landmark.x, toe_landmark.y, toe_landmark.z])

    # 3D distance in normalized units
    dist_norm = np.linalg.norm(toe - fingertip)

    # Convert to real-world — DO NOT TOUCH THESE VALUES!!!!
    dist_in = ((dist_norm - 0.11) * 27.27)

    # Determine category based on distance
    if fingertip[1] > toe[1]:
        # Fingertips past toes — positive
        if dist_in > 4:
            category = "Above Average"
            message = "Amazing! Your flexibility is Above Average — keep up the great work!"
        else:
            category = "Average"
            message = "Great effort! Your flexibility is Average — keep stretching daily to improve!"
    else:
        # Fingertips short of toes — negative
        if dist_in > 4:
            category = "Below Average"
            message = "Don't give up! Your flexibility is Below Average — daily stretching will help you improve!"
        else:
            category = "Average"
            message = "Great effort! Your flexibility is Average — keep stretching daily to improve!"

    # Print raw inches to terminal only — not shown to user
    print(f"[DEBUG] {leg_side} leg: {dist_in:.1f} inches | Category: {category}")

    return category, message

In [ ]:
# STEP 15 - Instruction Screen
import os
import time

def show_instruction_screen():

    cv2.namedWindow('Instructions', cv2.WINDOW_NORMAL)
    cv2.setWindowProperty('Instructions', cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

    # ── SCREEN 1: Written Instructions ──
    text_screen = np.ones((720, 1280, 3), dtype=np.uint8) * 255

    cv2.putText(text_screen, "Chair Sit and Reach Test - Instructions",
                (50, 70),
                cv2.FONT_HERSHEY_DUPLEX, 1.2, (30, 60, 150), 2)

    cv2.line(text_screen, (50, 110), (1230, 110), (30, 60, 150), 2)

    instructions = [
        "1. Sit on the EDGE of the chair",
        "2. Keep ONE foot flat on the floor",
        "3. Extend the OTHER leg straight forward",
        "4. Stack BOTH hands on top of each other",
        "5. Inhale deeply, then EXHALE and reach forward",
        "6. Hold the position for measurement",
    ]

    y = 180
    for line in instructions:
        cv2.putText(text_screen, line,
                    (50, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.4, (50, 50, 50), 2)
        y += 85

    cv2.rectangle(text_screen, (0, 640), (1280, 720), (30, 60, 150), -1)
    cv2.putText(text_screen, "Study the steps carefully - image guide coming up next...",
                (150, 690),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

    cv2.imshow('Instructions', text_screen)

    # Time-based exit only — no keypress in fullscreen Mac
    start = time.time()
    while time.time() - start < 15.0:
        cv2.waitKey(100)

    # ── SCREEN 2: Image Instructions ──
    notebook_dir = os.getcwd()
    image_path = os.path.join(notebook_dir, "visual_instruction.png")
    if not os.path.exists(image_path):
        image_path = "/Users/hinalsachpara/Desktop/CO-OP Docs/cv-flexibility-test-main/visual_instruction.png"

    instruction_img = cv2.imread(image_path)

    if instruction_img is None:
        print(f"Image not found at: {image_path}")
    else:
        instruction_img = cv2.resize(instruction_img, (1280, 720))
        cv2.rectangle(instruction_img, (0, 660), (1280, 720), (0, 0, 0), -1)
        cv2.putText(instruction_img,
                    "Starting in 5 seconds...",
                    (500, 700),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

        cv2.imshow('Instructions', instruction_img)

        # Time-based exit only
        start = time.time()
        while time.time() - start < 5.0:
            cv2.waitKey(100)

    cv2.destroyAllWindows()
    time.sleep(0.5)
    print("Instructions done! Starting countdown...")

In [ ]:
# STEP 16 - Error Messages
def show_error_message(frame, message):
    # ── WARNING — very top full width red bar ──
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (frame.shape[1], 75), (0, 0, 180), -1)
    cv2.addWeighted(overlay, 0.8, frame, 0.2, 0, frame)
    
    # Center the warning text
    warn_text = "WARNING: " + message
    warn_size = cv2.getTextSize(warn_text, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)[0]
    warn_x = (frame.shape[1] - warn_size[0]) // 2
    cv2.putText(frame, warn_text,
                (warn_x, 52),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    return frame

In [3]:
# STEP 17 - Result Screen
def show_result_screen(frozen_frame, measurement_result, window_name='Chair Sit and Reach Test'):
    if frozen_frame is None:
        frozen_frame = np.zeros((480, 640, 3), dtype=np.uint8)

    h, w = frozen_frame.shape[:2]

    blurred = cv2.GaussianBlur(frozen_frame, (55, 55), 0)
    dark_overlay = blurred.copy()
    cv2.rectangle(dark_overlay, (0, 0), (w, h), (0, 0, 0), -1)
    result_bg = cv2.addWeighted(blurred, 0.35, dark_overlay, 0.65, 0)

    cv2.rectangle(result_bg, (0, 0), (w, 90), (150, 60, 30), -1)
    header_text = "Chair Sit and Reach Test  -  Results"
    header_size = cv2.getTextSize(header_text, cv2.FONT_HERSHEY_DUPLEX, 1.1, 2)[0]
    header_x = (w - header_size[0]) // 2
    cv2.putText(result_bg, header_text, (header_x, 62),
                cv2.FONT_HERSHEY_DUPLEX, 1.1, (255, 255, 255), 2)

    cv2.line(result_bg, (40, 100), (w - 40, 100), (200, 200, 200), 1)

    complete_text = "TEST COMPLETE"
    complete_size = cv2.getTextSize(complete_text, cv2.FONT_HERSHEY_DUPLEX, 1.6, 3)[0]
    complete_x = (w - complete_size[0]) // 2
    cv2.putText(result_bg, complete_text, (complete_x, 170),
                cv2.FONT_HERSHEY_DUPLEX, 1.6, (0, 230, 0), 3)

    card_x1, card_y1 = int(w * 0.08), 200
    card_x2, card_y2 = int(w * 0.92), 430
    card_overlay = result_bg.copy()
    cv2.rectangle(card_overlay, (card_x1, card_y1), (card_x2, card_y2), (20, 20, 20), -1)
    cv2.addWeighted(card_overlay, 0.75, result_bg, 0.25, 0, result_bg)
    cv2.rectangle(result_bg, (card_x1, card_y1), (card_x2, card_y2), (80, 80, 80), 2)

    if measurement_result is not None:
        line1 = measurement_result[0]
        line2 = measurement_result[1]
    else:
        line1 = "No measurement recorded"
        line2 = "Please re-run the test"

    # line1 = category, line2 = message
    cv2.putText(result_bg, "Your Flexibility Result:",
                (card_x1 + 30, card_y1 + 50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.85, (180, 180, 180), 2)

    # Category — large cyan
    r1_size = cv2.getTextSize(line1, cv2.FONT_HERSHEY_DUPLEX, 1.5, 2)[0]
    r1_x = (w - r1_size[0]) // 2
    cv2.putText(result_bg, line1, (r1_x, card_y1 + 110),
                cv2.FONT_HERSHEY_DUPLEX, 1.5, (0, 230, 230), 3)

    cv2.line(result_bg, (card_x1 + 20, card_y1 + 135), (card_x2 - 20, card_y1 + 135), (80, 80, 80), 1)

    # Message — green
    r2_size = cv2.getTextSize(line2, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)[0]
    r2_x = (w - r2_size[0]) // 2
    cv2.putText(result_bg, line2, (r2_x, card_y1 + 185),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 210, 0), 2)

    cv2.rectangle(result_bg, (0, h - 70), (w, h), (30, 30, 30), -1)
    cv2.line(result_bg, (0, h - 70), (w, h - 70), (80, 80, 80), 1)
    prompt_text = "Press any key to exit"
    prompt_size = cv2.getTextSize(prompt_text, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)[0]
    prompt_x = (w - prompt_size[0]) // 2
    cv2.putText(result_bg, prompt_text, (prompt_x, h - 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (200, 200, 200), 2)

    for pulse_color in [(0, 200, 0), (0, 100, 0), (0, 200, 0)]:
        display = result_bg.copy()
        cv2.rectangle(display, (3, 3), (w - 3, h - 3), pulse_color, 3)
        cv2.imshow(window_name, display)
        if cv2.waitKey(300) != -1:
            return

    cv2.rectangle(result_bg, (3, 3), (w - 3, h - 3), (0, 200, 0), 2)
    cv2.imshow(window_name, result_bg)
    while True:
        if cv2.waitKey(50) != -1:
            break

In [ ]:
# STEP 18 - MAIN (RUN THIS LAST)
def main():

    main.last_instruction = None

    show_instruction_screen()

    cap = cv2.VideoCapture(0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    cv2.namedWindow('Chair Sit and Reach Test', cv2.WINDOW_NORMAL)
    cv2.setWindowProperty('Chair Sit and Reach Test', cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

    if not cap.isOpened():
        print("Error: Could not open camera")
        return

    test = FlexibilityTest()
    start_time = None
    test_start_time = None

    pre_countdown_phase = True
    full_body_confirmed = False
    full_body_confirm_time = None

    countdown_phase = False
    countdown_start_time = None
    countdown_step = 0

    last_good_frame = None
    result_screen_shown = False

    test.current_state = TestState.SITTING_CHECK
    test.phase = "validation"

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        last_good_frame = frame.copy()

        if test.current_state == TestState.COMPLETE and not result_screen_shown:
            result_screen_shown = True
            cap.release()
            show_result_screen(last_good_frame, test.measurement_result, 'Chair Sit and Reach Test')
            cv2.destroyAllWindows()
            return

        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)

        # ── PRE-COUNTDOWN ──
        if pre_countdown_phase:

            if not full_body_confirmed:
                prompt = "Position yourself fully in the camera frame"
                prompt_size = cv2.getTextSize(prompt, cv2.FONT_HERSHEY_DUPLEX, 1.6, 3)[0]
                prompt_x = (frame.shape[1] - prompt_size[0]) // 2
                cv2.rectangle(frame, (0, 75), (frame.shape[1], 145), (0, 0, 0), -1)
                cv2.putText(frame, prompt,
                            (prompt_x, 135),
                            cv2.FONT_HERSHEY_DUPLEX, 1.6, (255, 255, 255), 3)

                if results.pose_landmarks:
                    body_ok, _ = check_full_body(results.pose_landmarks.landmark)
                    if body_ok:
                        full_body_confirmed = True
                        full_body_confirm_time = time.time()
                    else:
                        frame = show_error_message(frame,
                            "Make sure head, body and feet are all visible!")
                else:
                    frame = show_error_message(frame,
                        "No body detected! Step into the camera frame.")

            else:
                confirm_msg = "Perfect! Get ready..."
                confirm_size = cv2.getTextSize(confirm_msg, cv2.FONT_HERSHEY_DUPLEX, 1.6, 3)[0]
                confirm_x = (frame.shape[1] - confirm_size[0]) // 2
                cv2.rectangle(frame, (0, 75), (frame.shape[1], 145), (0, 0, 0), -1)
                cv2.putText(frame, confirm_msg,
                            (confirm_x, 135),
                            cv2.FONT_HERSHEY_DUPLEX, 1.6, (0, 255, 0), 3)

                if time.time() - full_body_confirm_time >= 2.0:
                    pre_countdown_phase = False
                    countdown_phase = True

        # ── COUNTDOWN ──
        elif countdown_phase:
            countdown_messages = ["Get Ready: 3", "Get Ready: 2", "Get Ready: 1", "GO!"]
            audio_keys = ['countdown_3', 'countdown_2', 'countdown_1', 'countdown_go']

            # Draw countdown text first
            text = countdown_messages[countdown_step]
            text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_DUPLEX, 3, 4)[0]
            text_x = (frame.shape[1] - text_size[0]) // 2
            text_y = (frame.shape[0] + text_size[1]) // 2
            cv2.putText(frame, text, (text_x, text_y),
                       cv2.FONT_HERSHEY_DUPLEX, 5, (0, 255, 0), 6)

            if countdown_start_time is None:
                countdown_start_time = time.time()
                cv2.imshow('Chair Sit and Reach Test', frame)
                cv2.waitKey(1)
                time.sleep(0.3)
                tts_system.play_audio(audio_keys[countdown_step])

            if time.time() - countdown_start_time > 1.5:
                countdown_step += 1
                if countdown_step >= 4:
                    countdown_phase = False
                    test_start_time = time.time()
                else:
                    countdown_start_time = time.time()
                    cv2.imshow('Chair Sit and Reach Test', frame)
                    cv2.waitKey(1)
                    time.sleep(0.3)
                    tts_system.play_audio(audio_keys[countdown_step])


        
        # ── TEST PHASE ──
        else:
            if test_start_time is not None:
                elapsed = time.time() - test_start_time
                remaining = max(0, 30 - int(elapsed))

                if remaining == 0 and not result_screen_shown:
                    result_screen_shown = True
                    cap.release()
                    show_result_screen(last_good_frame, test.measurement_result, 'Chair Sit and Reach Test')
                    cv2.destroyAllWindows()
                    return

                timer_text = f"Time: {remaining}s"
                timer_size = cv2.getTextSize(timer_text, cv2.FONT_HERSHEY_DUPLEX, 1.8, 3)[0]
                timer_x = (frame.shape[1] - timer_size[0]) // 2
                cv2.rectangle(frame,
                    (0, frame.shape[0]-70),
                    (frame.shape[1], frame.shape[0]),
                    (0, 0, 0), -1)
                cv2.putText(frame, timer_text,
                            (timer_x, frame.shape[0]-15),
                            cv2.FONT_HERSHEY_DUPLEX, 1.8,
                            (0, 255, 0) if remaining > 10 else (0, 0, 255),
                            3)

            check_passed = False
            status_message = ""

            user_message = test.get_state_message()

            if user_message != main.last_instruction:
                speak(user_message)
                main.last_instruction = user_message

            text_size = cv2.getTextSize(user_message, cv2.FONT_HERSHEY_DUPLEX, 2.0, 3)[0]
            text_x = (frame.shape[1] - text_size[0]) // 2
            cv2.rectangle(frame, (0, 75), (frame.shape[1], 145), (0, 0, 0), -1)
            cv2.putText(frame, user_message,
                        (text_x, 135),
                        cv2.FONT_HERSHEY_DUPLEX, 2.0, (255, 255, 255), 3)

            if not results.pose_landmarks:
                frame = show_error_message(frame,
                    "No body detected! Please step into the camera frame.")

            elif results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark

                if test.current_state == TestState.SITTING_CHECK:
                    check_passed, status_message = check_sitting_position(landmarks)
                    if not check_passed:
                        frame = show_error_message(frame,
                            "Wrong position! Please sit on the edge of the chair.")

                elif test.current_state == TestState.FOOT_FLAT_CHECK:
                    check_passed, status_message = check_foot_flat_on_floor(landmarks)
                    if not check_passed:
                        frame = show_error_message(frame,
                            "Place one foot flat on the floor!")

                elif test.current_state == TestState.LEG_EXTENDED_CHECK:
                    check_passed, status_message = check_leg_extended(landmarks)
                    if not check_passed:
                        frame = show_error_message(frame,
                            "Extend your leg straight forward!")

                elif test.current_state == TestState.HANDS_POSITION_CHECK:
                    check_passed, status_message = check_hands_position(landmarks)
                    if not check_passed:
                        frame = show_error_message(frame,
                            "Place one hand on top of the other!")

                elif test.current_state == TestState.INHALE_PREPARATION:
                    check_passed, status_message = check_inhale(landmarks)

                elif test.current_state == TestState.FORWARD_REACH:
                    check_passed, status_message = check_forward_reach(landmarks)
                    if not check_passed:
                        frame = show_error_message(frame,
                            "Exhale and reach forward toward your toes!")

                elif test.current_state == TestState.HOLD_POSITION:
                    check_passed, status_message = check_forward_reach(landmarks)
                    if check_passed:
                        status_message = "Hold still! Measuring..."
                    else:
                        frame = show_error_message(frame,
                            "Keep reaching forward! Do not move!")

                elif test.current_state == TestState.MEASUREMENT:
                    test.measurement_result = measure_flexibility(landmarks)
                    check_passed = True

                color = (0, 255, 0) if check_passed else (0, 255, 255)

                status_size = cv2.getTextSize(status_message, cv2.FONT_HERSHEY_SIMPLEX, 1.0, 2)[0]
                status_x = (frame.shape[1] - status_size[0]) // 2
                cv2.rectangle(frame, (0, 150), (frame.shape[1], 195), (0, 0, 0), -1)
                cv2.putText(frame, status_message,
                            (status_x, 185),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)

                if check_passed:
                    if start_time is None:
                        start_time = time.time()
                    elif time.time() - start_time > (2 if test.current_state == TestState.HOLD_POSITION else 1):
                        if test.current_state != TestState.COMPLETE:
                            test.advance_state()
                            test.phase = "validation"
                        start_time = None
                else:
                    start_time = None

        cv2.imshow('Chair Sit and Reach Test', frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('r'):
            test.reset_test()
            test.current_state = TestState.SITTING_CHECK
            test.phase = "validation"
            pre_countdown_phase = True
            full_body_confirmed = False
            full_body_confirm_time = None
            countdown_phase = False
            countdown_start_time = None
            countdown_step = 0
            test_start_time = None
            result_screen_shown = False
        elif key == ord('m'):
            test.measure_test()

    cap.release()
    cv2.destroyAllWindows()

main()

In [ ]:
import subprocess

def test_video_countdown():
    import cv2
    import time
    
    # AUTO-FOCUS: Bring OpenCV window to front on Mac
    subprocess.Popen(['osascript', '-e', 
        'tell application "System Events" to set frontmost of every process whose name contains "Python" to true'])
    
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    time.sleep(2)
    
    if not cap.isOpened():
        print("Error: Could not open camera")
        return
    
    # WINDOW WARMUP
    print("Warming up display window...")
    for _ in range(30):
        ret, frame = cap.read()
        if ret:
            cv2.putText(frame, "Get Ready...", (100, 250),
                       cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 4)
            cv2.imshow('Flexibility Test', frame)
            cv2.waitKey(1)
    
    # Auto-focus window after warmup
    subprocess.Popen(['osascript', '-e',
        'tell application "System Events" to set frontmost of every process whose name contains "Python" to true'])
    
    print("Starting countdown...")
    messages = ["Get Ready: 3", "Get Ready: 2", "Get Ready: 1", "GO!"]
    audio_keys = ['countdown_3', 'countdown_2', 'countdown_1', 'countdown_go']
    
    for step in range(4):
        tts_system.play_audio(audio_keys[step])
        print(f"Playing: {messages[step]}")
        
        start = time.time()
        while time.time() - start < 1.0:
            ret, frame = cap.read()
            if not ret:
                continue
            cv2.putText(frame, messages[step], (100, 250),
                       cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 4)
            cv2.imshow('Flexibility Test', frame)
            cv2.waitKey(1)
    
    print("Countdown complete!")
    end_time = time.time() + 5
    while time.time() < end_time:
        ret, frame = cap.read()
        if not ret:
            continue
        cv2.putText(frame, "Test Starting!", (100, 250),
                   cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 4)
        cv2.imshow('Flexibility Test', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    print("Video test complete!")

test_video_countdown()

In [ ]:
import cv2
import time
import subprocess

def main():

    # Auto-focus window
    subprocess.Popen(['osascript', '-e',
        'tell application "System Events" to set frontmost of every process whose name contains "Python" to true'])

    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    time.sleep(2)

    if not cap.isOpened():
        print("Error: Could not open camera")
        return

    # ---- INSTRUCTION SCREEN BEFORE COUNTDOWN ----
    show_instruction_screen("/Users/hinalsachpara/Desktop/CO-OP Docs/cv-flexibility-test-main/Instruction Picture/merged_instruction.png")

    # Window warmup
    print("Warming up camera...")
    warmup_start = time.time()
    while time.time() - warmup_start < 2.0:
        ret, frame = cap.read()
        if ret:
            cv2.putText(frame, "Get Ready...", (100, 250),
                       cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 4)
            cv2.imshow('Chair Sit and Reach Test', frame)
            cv2.waitKey(1)

    # Auto focus after warmup
    subprocess.Popen(['osascript', '-e',
        'tell application "System Events" to set frontmost of every process whose name contains "Python" to true'])

    # COUNTDOWN 3-2-1-GO
    messages = ["Get Ready: 3", "Get Ready: 2", "Get Ready: 1", "GO!"]
    audio_keys = ['countdown_3', 'countdown_2', 'countdown_1', 'countdown_go']

    for step in range(4):
        tts_system.play_audio(audio_keys[step])
        start = time.time()
        while time.time() - start < 1.0:
            ret, frame = cap.read()
            if not ret:
                continue
            cv2.putText(frame, messages[step], (100, 250),
                       cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 4)
            cv2.imshow('Chair Sit and Reach Test', frame)
            cv2.waitKey(1)

    # INSTRUCTION STAGES WITH ERROR DETECTION
    # Each stage shows instruction text AND checks for errors
    stages = [
        ("Please sit on edge of chair", "instruction_sit", "SITTING_CHECK"),
        ("Extend one leg forward", "instruction_extend", "LEG_EXTENDED_CHECK"),
        ("Place foot flat on floor", None, "FOOT_FLAT_CHECK"),
        ("Stack your hands on chest", None, "HANDS_POSITION_CHECK"),
        ("Inhale deeply...", None, "INHALE_PREPARATION"),
        ("Reach forward toward toes", None, "FORWARD_REACH"),
        ("Hold your position!", "instruction_hold", "HOLD_POSITION"),
    ]

    for instruction, audio_key, state_name in stages:
        if audio_key:
            tts_system.play_audio(audio_key)

        print(f"Stage: {instruction}")

        start = time.time()
        while time.time() - start < 3.0:
            ret, frame = cap.read()
            if not ret:
                continue

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(rgb)

            # ---- ERROR DETECTION STARTS HERE ----
            if not results.pose_landmarks:
                # No body detected at all
                frame = show_error_message(frame,
                    "No body detected! Please step into the camera frame.")
            else:
                landmarks = results.pose_landmarks.landmark

                # Draw skeleton on frame
                mp_drawing.draw_landmarks(
                    frame, results.pose_landmarks,
                    mp_pose.POSE_CONNECTIONS)

                # Check specific error for each stage
                if state_name == "SITTING_CHECK":
                    check_passed, _ = check_sitting_position(landmarks)
                    if not check_passed:
                        frame = show_error_message(frame,
                            "Wrong position! Please sit on the edge of the chair.")

                elif state_name == "FOOT_FLAT_CHECK":
                    check_passed, _ = check_foot_flat_on_floor(landmarks)
                    if not check_passed:
                        frame = show_error_message(frame,
                            "Place one foot flat on the floor!")

                elif state_name == "LEG_EXTENDED_CHECK":
                    check_passed, _ = check_leg_extended(landmarks)
                    if not check_passed:
                        frame = show_error_message(frame,
                            "Extend your leg straight forward!")

                elif state_name == "HANDS_POSITION_CHECK":
                    check_passed, _ = check_hands_position(landmarks)
                    if not check_passed:
                        frame = show_error_message(frame,
                            "Place one hand on top of the other!")

                elif state_name == "FORWARD_REACH":
                    check_passed, _ = check_forward_reach(landmarks)
                    if not check_passed:
                        frame = show_error_message(frame,
                            "Exhale and reach forward toward your toes!")

                # Always check full body visibility
                full_body_passed, _ = check_full_body(landmarks)
                if not full_body_passed:
                    frame = show_error_message(frame,
                        "Adjust camera! Make sure your full body is visible.")
            # ---- ERROR DETECTION ENDS HERE ----

            # Always show the instruction text
            cv2.putText(frame, instruction, (30, 120),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            cv2.putText(frame, f"Stage: {state_name}", (30, 160),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

            cv2.imshow('Chair Sit and Reach Test', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                cap.release()
                cv2.destroyAllWindows()
                return

    # MEASUREMENT
    print("Measuring flexibility...")
    tts_system.play_audio('instruction_hold')
    result_text = "Measuring..."

    start = time.time()
    while time.time() - start < 3.0:
        ret, frame = cap.read()
        if not ret:
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb)

        if not results.pose_landmarks:
            frame = show_error_message(frame,
                "No body detected! Please step into the camera frame.")
        else:
            mp_drawing.draw_landmarks(
                frame, results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS)
            result_text = measure_flexibility(results.pose_landmarks.landmark)

        cv2.putText(frame, "Measuring...", (30, 120),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255),

In [ ]:
print(tts_system.audio_files.keys())

In [ ]:
def play_audio_fast(audio_key):
    try:
        audio_file = tts_system.audio_files[audio_key]
        if os.path.exists(audio_file):
            pygame.mixer.music.load(audio_file)
            pygame.mixer.music.play()
    except Exception as e:
        print(f"Audio playback error: {e}")

tts_system.play_audio = play_audio_fast

In [ ]:
import inspect
print(inspect.getsource(tts_system.play_audio))

In [ ]:
import pygame
pygame.mixer.init()
pygame.mixer.music.load(list(tts_system.audio_files.values())[0])
pygame.mixer.music.play()
import time
time.sleep(2)
print("Audio test done!")
print("Available audio keys:", list(tts_system.audio_files.keys()))

In [ ]:
test_temp = FlexibilityTest()
for state in TestState:
    test_temp.current_state = state
    print(f"{state.name}: '{test_temp.get_state_message()}'")

In [ ]:
import os
audio_dir = "/Users/hinalsachpara/Desktop/CO-OP Docs/cv-flexibility-test-main/audio"
files = os.listdir(audio_dir)
for f in sorted(files):
    print(f)

In [ ]:
test_temp = FlexibilityTest()
for state in TestState:
    test_temp.current_state = state
    preview = test_temp.get_preview_message()
    instruction = test_temp.get_state_message()
    print(f"{state.name}:")
    print(f"  Preview:     '{preview}'")
    print(f"  Instruction: '{instruction}'")
    print()

In [ ]:
# hand thershold testing 
# Test 3 positions to set the right threshold
import cv2, time, math
cap = cv2.VideoCapture(0)

for i in range(60):
    ret, frame = cap.read()
    if not ret: break
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    r = pose.process(rgb)
    if r.pose_landmarks:
        lm = r.pose_landmarks.landmark
        lw = lm[mp_pose.PoseLandmark.LEFT_WRIST.value]
        li = lm[mp_pose.PoseLandmark.LEFT_INDEX.value]
        lp = lm[mp_pose.PoseLandmark.LEFT_PINKY.value]
        rw = lm[mp_pose.PoseLandmark.RIGHT_WRIST.value]
        ri = lm[mp_pose.PoseLandmark.RIGHT_INDEX.value]
        rp = lm[mp_pose.PoseLandmark.RIGHT_PINKY.value]
        lx=(lw.x+li.x+lp.x)/3; ly=(lw.y+li.y+lp.y)/3; lz=(lw.z+li.z+lp.z)/3
        rx=(rw.x+ri.x+rp.x)/3; ry=(rw.y+ri.y+rp.y)/3; rz=(rw.z+ri.z+rp.z)/3
        dist = math.sqrt((lx-rx)**2+(ly-ry)**2+(lz-rz)**2)

        if i < 20:   label = "HANDS APART"
        elif i < 40: label = "HANDS STACKED"
        else:        label = "ARMS EXTENDED FORWARD"

        print(f"{label} | Distance: {dist:.4f}")
    time.sleep(0.2)

cap.release()
print("Done")

In [ ]:
# hand thershold testing 
cap.release()
cv2.destroyAllWindows()

In [ ]:
# hand thershold testing 
import cv2, time, math

cap = cv2.VideoCapture(0)
print("Camera open — hold your hands in position and watch the numbers...")
print("Press Q in the terminal or stop the cell to quit\n")

for _ in range(200):
    ret, frame = cap.read()
    if not ret:
        print("Camera not reading")
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    r = pose.process(rgb)

    if r.pose_landmarks:
        lm = r.pose_landmarks.landmark

        # Get hand landmarks
        lw = lm[mp_pose.PoseLandmark.LEFT_WRIST.value]
        li = lm[mp_pose.PoseLandmark.LEFT_INDEX.value]
        lp = lm[mp_pose.PoseLandmark.LEFT_PINKY.value]

        rw = lm[mp_pose.PoseLandmark.RIGHT_WRIST.value]
        ri = lm[mp_pose.PoseLandmark.RIGHT_INDEX.value]
        rp = lm[mp_pose.PoseLandmark.RIGHT_PINKY.value]

        # Calculate centers
        lx = (lw.x + li.x + lp.x) / 3
        ly = (lw.y + li.y + lp.y) / 3
        lz = (lw.z + li.z + lp.z) / 3

        rx = (rw.x + ri.x + rp.x) / 3
        ry = (rw.y + ri.y + rp.y) / 3
        rz = (rw.z + ri.z + rp.z) / 3

        dist = math.sqrt((lx-rx)**2 + (ly-ry)**2 + (lz-rz)**2)
        passed = dist < 0.1

        print(f"Distance: {dist:.4f} | Threshold: 0.10 | {'✅ PASS' if passed else '❌ FAIL'}")
    else:
        print("No body detected")

    time.sleep(0.2)

cap.release()
print("Done")

In [ ]:
# debug code 
import cv2, time
cap = cv2.VideoCapture(0)
print("Reach forward and hold — watching values...")

for _ in range(50):
    ret, frame = cap.read()
    if not ret: break
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    r = pose.process(rgb)
    if r.pose_landmarks:
        lm = r.pose_landmarks.landmark
        lw = lm[mp_pose.PoseLandmark.LEFT_WRIST.value]
        rw = lm[mp_pose.PoseLandmark.RIGHT_WRIST.value]
        lk = lm[mp_pose.PoseLandmark.LEFT_KNEE.value]
        rk = lm[mp_pose.PoseLandmark.RIGHT_KNEE.value]
        la = lm[mp_pose.PoseLandmark.LEFT_ANKLE.value]
        ra = lm[mp_pose.PoseLandmark.RIGHT_ANKLE.value]

        hand_y = (lw.y + rw.y) / 2
        hand_x = (lw.x + rw.x) / 2

        left_angle  = calculate_angle_3d(lm[mp_pose.PoseLandmark.LEFT_HIP.value], lk, la)
        right_angle = calculate_angle_3d(lm[mp_pose.PoseLandmark.RIGHT_HIP.value], rk, ra)

        if left_angle > right_angle:
            ext_ankle = la; ext_knee = lk
        else:
            ext_ankle = ra; ext_knee = rk

        below_knee = hand_y > ext_knee.y
        x_diff     = abs(hand_x - ext_ankle.x)
        near_foot  = x_diff < 0.2

        print(f"below_knee={below_knee} | x_diff={x_diff:.4f} near_foot={near_foot} | {'✅ PASS' if (below_knee and near_foot) else '❌ FAIL'}")
    time.sleep(0.2)

cap.release()
#print("Done")